# Lecture 7: Convolutional Networks

*Notebook for UMich EECS 498 (Justin Johnson, Fall 2019), Lecture 7, with material from Stanford CS231n 2026, Lectures 5–6.*

So far, our classifiers flatten an image into a long list of numbers, so they have **no idea which pixels are neighbors**. Convolutional networks (CNNs) use layers designed for images.

**What you'll learn**
1. The **convolution** layer: sliding a small filter over the image.
2. Output sizes, **padding**, **stride**, and **receptive fields**.
3. **Pooling** layers, and why CNNs tolerate small shifts.
4. The classic CNN pattern (LeNet-5), and training a small CNN ourselves.
5. **Batch normalization** and its cousins.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from sklearn.datasets import load_digits

## 1. Fully connected vs. convolution

**Fully connected layer:** every output looks at **every** pixel. For a 224×224×3 image and 4,096 outputs, that's a huge number of weights:

In [ ]:
print(f"fully connected, 224x224x3 -> 4096: {224 * 224 * 3 * 4096:,} weights")
print(f"convolution, 64 filters of size 3x3x3: {64 * (3 * 3 * 3 + 1):,} weights")

**Convolution layer:** a small **filter** (say 3×5×5) slides over the image. At each position it computes **one dot product** between the filter and the patch under it. Two big ideas:
* **local connections:** each output looks only at a small patch;
* **shared weights:** the *same* filter is used at every position.

## 2. Convolution by hand

A 4×4 image and a 2×2 filter. Place the filter on the top-left 2×2 patch, multiply element by element, and add up:

In [ ]:
image = np.array([[1, 2, 0, 1],
                  [0, 1, 3, 2],
                  [2, 1, 0, 1],
                  [1, 0, 2, 3]])
filt = np.array([[1, 0],
                 [0, -1]])

top_left_patch = image[0:2, 0:2]
print("top-left patch:\n", top_left_patch)
print("patch * filter, summed:", np.sum(top_left_patch * filt), "  (1*1 + 2*0 + 0*0 + 1*(-1) = 0)")

Now slide the filter to every position. The filter fits in $4 - 2 + 1 = 3$ positions each way, so the output is 3×3:

In [ ]:
def convolve(image, filt):
    # Slide `filt` over `image` (no padding, stride 1) and take a dot product at each position.
    k = filt.shape[0]
    out_size = image.shape[0] - k + 1
    out = np.zeros((out_size, out_size))
    for i in range(out_size):
        for j in range(out_size):
            patch = image[i:i + k, j:j + k]
            out[i, j] = np.sum(patch * filt)
    return out

print(convolve(image, filt))

(Technically this is "cross-correlation"; real convolution flips the filter first. Since filters are learned, it makes no difference, and all deep learning libraries do it this way.)

### Real convolution layers: many channels, many filters
* A color image has **3 channels**, so each filter is also 3 deep (e.g., 3×5×5). **Filters always extend through all the input channels.**
* A layer has **many filters**. Each one produces one **activation map**; we stack them.
* So an input of $C_{in}\times H\times W$ with $C_{out}$ filters of size $C_{in}\times K\times K$ gives an output of $C_{out}\times H'\times W'$.

Here's the general version, with **padding** and **stride** (explained in Section 3):

In [ ]:
def conv_layer(x, filters, biases, stride=1, pad=0):
    # x: (C_in, H, W)   filters: (C_out, C_in, K, K)   biases: (C_out,)   ->   (C_out, H_out, W_out)
    C_out, C_in, K, _ = filters.shape
    x = np.pad(x, ((0, 0), (pad, pad), (pad, pad)))          # add zeros around each channel
    H_out = (x.shape[1] - K) // stride + 1
    W_out = (x.shape[2] - K) // stride + 1
    out = np.zeros((C_out, H_out, W_out))
    for f in range(C_out):                                    # for each filter...
        for i in range(H_out):                                # ...at each position...
            for j in range(W_out):
                patch = x[:, i*stride:i*stride + K, j*stride:j*stride + K]   # C_in x K x K
                out[f, i, j] = np.sum(patch * filters[f]) + biases[f]       # ...one dot product
    return out

rng = np.random.default_rng(0)
x_rgb = rng.normal(size=(3, 32, 32))                          # a fake 3-channel 32x32 image
filters = rng.normal(size=(6, 3, 5, 5))                       # 6 filters, each 3x5x5
print("output shape:", conv_layer(x_rgb, filters, np.zeros(6)).shape, "  (6 activation maps of 28x28)")

## 3. Spatial sizes: padding and stride

* Input $W$, filter $K$ → output $W - K + 1$. **Problem: the image shrinks with every layer.**
* **Padding** $P$: add zeros around the border. Output $W - K + 1 + 2P$. With $P = (K - 1)/2$ the size stays the same ("same padding").
* **Stride** $S$: move the filter $S$ pixels at a time.

$$\text{output size} = \frac{W - K + 2P}{S} + 1$$

In [ ]:
def output_size(W, K, P, S):
    return (W - K + 2 * P) // S + 1

print("7x7 input, 3x3 filter:             ", output_size(7, 3, 0, 1))
print("7x7 input, 3x3 filter, pad 1:      ", output_size(7, 3, 1, 1), "(same size)")
print("7x7 input, 3x3 filter, stride 2:   ", output_size(7, 3, 0, 2))

**The lecture's example:** input 3×32×32, ten 5×5 filters, stride 1, pad 2.

In [ ]:
size = output_size(32, 5, 2, 1)
params = 10 * (3 * 5 * 5 + 1)                      # each filter: 3*5*5 weights + 1 bias
multiply_adds = (10 * size * size) * (3 * 5 * 5)   # number of outputs x work per output
print(f"output: 10 x {size} x {size}")
print(f"learnable parameters: {params}")
print(f"multiply-adds: {multiply_adds:,}")

**Common settings:** 3×3 filters with pad 1 and stride 1 (keeps the size), 1×1 filters, and 3×3 with stride 2 (halves the size).

### Stacking convolutions
**Q: What happens if we put two convolutions in a row?** A convolution is a linear operation, so two of them combine into **one bigger convolution**: no extra power. We need a **ReLU** in between, just like in Lecture 5. Check: two 3×3 convolutions equal one 5×5 convolution.

In [ ]:
from scipy.signal import convolve2d
img = rng.normal(size=(12, 12))
f1 = rng.normal(size=(3, 3))
f2 = rng.normal(size=(3, 3))
two_convs = convolve(convolve(img, f1), f2)
combined_filter = convolve2d(f1, f2)                 # the single 5x5 filter that does both
one_conv = convolve(img, combined_filter)
print("two 3x3 convs == one 5x5 conv:", np.allclose(two_convs, one_conv))

### Receptive fields
Each output of a $K\times K$ convolution depends on a $K\times K$ patch of its input (its **receptive field**). Stack layers and it grows: **each extra layer adds $K - 1$**, so $L$ layers of 3×3 see $1 + 2L$ pixels across. For a big image, that takes a lot of layers, which is one reason networks also **downsample** (with stride or pooling): after downsampling, every layer covers more of the original image.

In [ ]:
for layers in [1, 2, 3, 5, 10]:
    print(f"{layers:2d} layers of 3x3 convs: receptive field {1 + 2 * layers} x {1 + 2 * layers}")

### 1×1 convolutions
A 1×1 filter looks at one pixel, but **all its channels**. So a 1×1 convolution is a small fully connected layer applied at every pixel. It's a cheap way to change the number of channels (used heavily in Lecture 8's architectures).

## 4. Pooling

**Pooling** shrinks each channel separately, with no learnable weights. **Max pooling** with a 2×2 window and stride 2 keeps the largest value in each 2×2 block. The lecture's example:

In [ ]:
x_pool = np.array([[1, 1, 2, 4],
                   [5, 6, 7, 8],
                   [3, 2, 1, 0],
                   [1, 2, 3, 4]])

def max_pool(x, size=2):
    H, W = x.shape
    out = np.zeros((H // size, W // size))
    for i in range(H // size):
        for j in range(W // size):
            block = x[i*size:(i + 1)*size, j*size:(j + 1)*size]
            out[i, j] = block.max()
    return out

print("max pooling:\n", max_pool(x_pool), "   (slides: [[6, 8], [3, 4]])")

Max pooling gives **tolerance to small shifts**: if a feature moves a little within its 2×2 window, the max doesn't change. (This is Hubel and Wiesel's complex cell from Lecture 1!)

### Convolution is "translation equivariant"
**Shift the image, then convolve = convolve, then shift the result.** The same filter is applied everywhere, so a pattern produces the same response wherever it is; the response just moves with it.

In [ ]:
picture = np.zeros((16, 16))
picture[5:9, 4:7] = rng.random((4, 3))                          # a small object
shifted_picture = np.roll(picture, (3, 2), axis=(0, 1))          # move it 3 down, 2 right

response = convolve(picture, f1)
response_of_shifted = convolve(shifted_picture, f1)
print("conv(shift(x)) == shift(conv(x)):", np.allclose(response_of_shifted, np.roll(response, (3, 2), axis=(0, 1))))

## 5. A classic CNN: LeNet-5

The classic pattern: **[Conv → ReLU → Pool] × N, then flatten, then fully connected layers.** LeNet-5 (LeCun et al., 1998), as presented in the lecture:

In [ ]:
shape = (1, 28, 28)
print(f"{'layer':30s} {'output shape':>14s} {'parameters':>11s}")
print(f"{'input':30s} {str(shape):>14s}")

c, s = 1, 28
layers = [("conv 20 filters 5x5, pad 2", 20), ("max pool 2x2", None), ("conv 50 filters 5x5, pad 2", 50), ("max pool 2x2", None)]
for name, n_filters in layers:
    if n_filters:                                   # a conv layer
        p = n_filters * (c * 5 * 5 + 1)
        c, s = n_filters, output_size(s, 5, 2, 1)
    else:                                           # a pooling layer
        p = 0
        s = s // 2
    print(f"{name:30s} {str((c, s, s)):>14s} {p:11,d}")

flat = c * s * s
print(f"{'flatten':30s} {str((flat,)):>14s}")
print(f"{'fully connected -> 500':30s} {str((500,)):>14s} {500 * (flat + 1):11,d}")
print(f"{'fully connected -> 10':30s} {str((10,)):>14s} {10 * (500 + 1):11,d}")

**Notice the trend:** as we go deeper, the **spatial size shrinks** and the **number of channels grows**. The network trades "where" for "what".

## 6. Training a small CNN on the digits

Let's train a tiny CNN on our 8×8 digits and compare it with an MLP:

**conv (8 filters, 3×3, pad 1) → ReLU → max pool 2×2 → fully connected → 10 scores**

We need the backward pass of each layer. All of them follow Lecture 6's rule (upstream × local):
* **Fully connected:** as in Lecture 6.
* **Max pool:** the gradient goes only to the pixel that was the max (a "router").
* **Conv:** each filter weight was used at every position, so its gradient **adds up** (upstream × input patch) over all positions (the copy rule).

First the data, batched as `(N, 8, 8)`:

In [ ]:
digits = load_digits()
images = digits.images / 16.0 - 0.3          # scale to roughly centered values
labels = digits.target
order = np.random.default_rng(0).permutation(len(images))
train_idx, test_idx = order[:1300], order[1300:]
X_train, y_train = images[train_idx], labels[train_idx]
X_test, y_test = images[test_idx], labels[test_idx]

The forward pass, written for a **batch** of images. The loops go over filter positions (8×8 = 64 of them); NumPy handles the whole batch at once.

In [ ]:
def cnn_forward(params, X):
    # X: (N, 8, 8). Returns the scores and everything the backward pass needs.
    N = len(X)
    W, b = params["conv_W"], params["conv_b"]                  # (8, 3, 3) and (8,)
    X_padded = np.pad(X, ((0, 0), (1, 1), (1, 1)))
    conv = np.zeros((N, 8, 8, 8))                              # (images, filters, rows, cols)
    for i in range(8):
        for j in range(8):
            patch = X_padded[:, i:i + 3, j:j + 3]              # (N, 3, 3): the patch at this position, for every image
            for f in range(8):
                conv[:, f, i, j] = np.sum(patch * W[f], axis=(1, 2)) + b[f]
    relu = np.maximum(0, conv)
    pooled = relu.reshape(N, 8, 4, 2, 4, 2).max(axis=(3, 5))   # 2x2 max pooling: (N, 8, 4, 4)
    flat = pooled.reshape(N, 128)
    scores = flat @ params["fc_W"] + params["fc_b"]
    cache = (X_padded, conv, relu, pooled, flat)
    return scores, cache

(The pooling line splits each 8×8 map into 2×2 blocks and takes the max of each block, the same thing as `max_pool` above, for the whole batch at once.)

The backward pass, layer by layer in reverse:

In [ ]:
def cnn_backward(params, cache, d_scores):
    X_padded, conv, relu, pooled, flat = cache
    N = len(d_scores)
    grads = {}
    # fully connected layer
    grads["fc_W"] = flat.T @ d_scores
    grads["fc_b"] = d_scores.sum(axis=0)
    d_pooled = (d_scores @ params["fc_W"].T).reshape(N, 8, 4, 4)
    # max pooling: send each gradient to the position that was the max in its 2x2 block.
    # Careful: blank areas give several identical values, so a block can have a TIE for the max.
    # Then we split the gradient between the tied positions (otherwise we'd count it twice).
    is_max = relu == np.repeat(np.repeat(pooled, 2, axis=2), 2, axis=3)
    ties = is_max.reshape(N, 8, 4, 2, 4, 2).sum(axis=(3, 5))                 # how many maxes in each block
    share = is_max / np.repeat(np.repeat(ties, 2, axis=2), 2, axis=3)
    d_relu = share * np.repeat(np.repeat(d_pooled, 2, axis=2), 2, axis=3)
    # ReLU
    d_conv = d_relu * (conv > 0)
    # convolution: each weight's gradient adds up (upstream x input patch) over every position
    grads["conv_W"] = np.zeros_like(params["conv_W"])
    grads["conv_b"] = d_conv.sum(axis=(0, 2, 3))
    for i in range(8):
        for j in range(8):
            patch = X_padded[:, i:i + 3, j:j + 3]
            for f in range(8):
                grads["conv_W"][f] += np.sum(d_conv[:, f, i, j][:, None, None] * patch, axis=0)
    return grads

The loss (softmax, as always), and a gradient check on one conv weight:

In [ ]:
def softmax_loss(scores, y):
    shifted = scores - scores.max(axis=1, keepdims=True)
    probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
    loss = -np.mean(np.log(probs[np.arange(len(y)), y]))
    probs[np.arange(len(y)), y] -= 1
    return loss, probs / len(y)

def init_cnn(seed=0):
    rng = np.random.default_rng(seed)
    return {"conv_W": rng.normal(0, np.sqrt(2 / 9), size=(8, 3, 3)), "conv_b": np.zeros(8),
            "fc_W": rng.normal(0, np.sqrt(2 / 128), size=(128, 10)), "fc_b": np.zeros(10)}

params = init_cnn(seed=1)
scores, cache = cnn_forward(params, X_train[:10])
loss, d_scores = softmax_loss(scores, y_train[:10])
grads = cnn_backward(params, cache, d_scores)

original = params["conv_W"][2, 1, 1]
params["conv_W"][2, 1, 1] = original + 1e-6; loss_plus, _ = softmax_loss(cnn_forward(params, X_train[:10])[0], y_train[:10])
params["conv_W"][2, 1, 1] = original - 1e-6; loss_minus, _ = softmax_loss(cnn_forward(params, X_train[:10])[0], y_train[:10])
params["conv_W"][2, 1, 1] = original
print(f"conv weight gradient: backprop {grads['conv_W'][2, 1, 1]:.6f}, numerical {(loss_plus - loss_minus) / 2e-6:.6f}")

Now train it with minibatch SGD with momentum (Lecture 4):

In [ ]:
def train_cnn(epochs=15, lr=0.02, batch_size=32):
    params = init_cnn()
    velocity = {name: np.zeros_like(value) for name, value in params.items()}
    rng = np.random.default_rng(0)
    for epoch in range(epochs):
        order_epoch = rng.permutation(len(X_train))
        for start in range(0, len(X_train), batch_size):
            batch = order_epoch[start:start + batch_size]
            scores, cache = cnn_forward(params, X_train[batch])
            loss, d_scores = softmax_loss(scores, y_train[batch])
            grads = cnn_backward(params, cache, d_scores)
            for name in params:
                velocity[name] = 0.9 * velocity[name] - lr * grads[name]
                params[name] += velocity[name]
    return params

cnn_params = train_cnn()
cnn_accuracy = np.mean(np.argmax(cnn_forward(cnn_params, X_test)[0], axis=1) == y_test)
print(f"small CNN test accuracy: {cnn_accuracy:.1%}")

What did the filters learn? Small local patterns, like edges and strokes:

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for f in range(8):
    axes[f].imshow(cnn_params["conv_W"][f], cmap="RdBu_r")
    axes[f].axis("off")
plt.suptitle("the 8 learned 3x3 filters", fontsize=9)
plt.show()

**Compare with an MLP** (64 → 100 → 10), especially on test digits **shifted by one pixel**:

In [ ]:
def train_mlp(epochs=30, lr=0.02, batch_size=32):
    rng = np.random.default_rng(0)
    W1 = rng.normal(0, np.sqrt(2 / 64), size=(64, 100)); b1 = np.zeros(100)
    W2 = rng.normal(0, np.sqrt(2 / 100), size=(100, 10)); b2 = np.zeros(10)
    X_flat = X_train.reshape(-1, 64)
    for epoch in range(epochs):
        order_epoch = rng.permutation(len(X_flat))
        for start in range(0, len(X_flat), batch_size):
            batch = order_epoch[start:start + batch_size]
            z = X_flat[batch] @ W1 + b1; h = np.maximum(0, z)
            loss, d_scores = softmax_loss(h @ W2 + b2, y_train[batch])
            d_z = (d_scores @ W2.T) * (z > 0)
            W2 -= lr * 5 * h.T @ d_scores; b2 -= lr * 5 * d_scores.sum(0)
            W1 -= lr * 5 * X_flat[batch].T @ d_z; b1 -= lr * 5 * d_z.sum(0)
    return lambda X: np.argmax(np.maximum(0, X.reshape(len(X), 64) @ W1 + b1) @ W2 + b2, axis=1)

mlp_predict = train_mlp()
X_test_shifted = np.array([ndimage.shift(img, (0, 1), order=0, cval=-0.3) for img in X_test])

cnn_params_count = sum(v.size for v in cnn_params.values())
print(f"{'model':8s} {'parameters':>11s} {'test':>7s} {'shifted 1 px':>13s}")
print(f"{'CNN':8s} {cnn_params_count:11,d} {cnn_accuracy:7.1%} "
      f"{np.mean(np.argmax(cnn_forward(cnn_params, X_test_shifted)[0], axis=1) == y_test):13.1%}")
print(f"{'MLP':8s} {64 * 100 + 100 + 100 * 10 + 10:11,d} {np.mean(mlp_predict(X_test) == y_test):7.1%} "
      f"{np.mean(mlp_predict(X_test_shifted) == y_test):13.1%}")

**What we see:** the CNN has far fewer parameters, does nearly as well on normal digits, and holds up better when the digits shift: **shared filters + pooling** respond to the same strokes wherever they are.

## 7. Batch normalization

**Deep networks are hard to train.** **Batch normalization** (Ioffe & Szegedy, 2015) helps: after a layer, **normalize each feature to mean 0 and variance 1, using the current minibatch's statistics**:

$$\hat x = \frac{x - \text{mean}}{\sqrt{\text{variance} + \epsilon}}, \qquad y = \gamma\,\hat x + \beta$$

$\gamma$ and $\beta$ are **learned**, so the network can undo the normalization if that's better (with $\gamma = \sqrt{\text{variance}}$ and $\beta = \text{mean}$, it's the identity).

By hand: one feature over a batch of 4 examples, values $[1, 2, 3, 6]$:

In [ ]:
batch = np.array([1.0, 2.0, 3.0, 6.0])
mean = batch.mean()                   # 3
variance = batch.var()                # (4 + 1 + 0 + 9) / 4 = 3.5
normalized = (batch - mean) / np.sqrt(variance + 1e-5)
print("mean:", mean, "  variance:", variance)
print("normalized:", normalized.round(3), "-> mean", round(normalized.mean(), 3) + 0.0, ", std", round(normalized.std(), 3))

**At test time** there might be only one image, and the prediction shouldn't depend on which other images happen to be in the batch. So during training we keep a **running average** of the mean and variance, and at test time we use those fixed numbers.

**This is a very common source of bugs!** If you forget to switch to test mode, a batch of one image gets normalized by its *own* statistics, and every feature becomes 0:

In [ ]:
single_example = np.array([[5.0, -1.0, 2.0]])               # one example, 3 features

training_mode = (single_example - single_example.mean(axis=0)) / np.sqrt(single_example.var(axis=0) + 1e-5)
print("training mode, batch of 1:", training_mode, " <- all zeros, whatever the input!")

running_mean, running_var = np.array([2.0, 0.0, 1.0]), np.array([9.0, 4.0, 1.0])   # collected during training
test_mode = (single_example - running_mean) / np.sqrt(running_var + 1e-5)
print("test mode (running stats):", test_mode.round(3))

(In PyTorch that's `model.eval()` before testing and `model.train()` before training.)

**Where it goes:** usually right after a convolution or fully connected layer, before the ReLU. **For conv layers**, each channel is normalized over the batch *and* all spatial positions.

**Why use it?** Deep networks become much easier to train, you can use higher learning rates, training is less sensitive to initialization, and it even regularizes a bit. At test time it's just a fixed scale and shift, so it can be folded into the previous layer for free.

### Layer norm, instance norm, group norm
These all do the same normalization, just over **different axes** of an activation tensor of shape (N, C, H, W):

In [ ]:
def normalize(x, axes):
    return (x - x.mean(axis=axes, keepdims=True)) / np.sqrt(x.var(axis=axes, keepdims=True) + 1e-5)

activations = rng.normal(3, 2, size=(4, 6, 5, 5))         # (N, C, H, W)
batch_norm = normalize(activations, axes=(0, 2, 3))      # each channel, over the batch and space
layer_norm = normalize(activations, axes=(1, 2, 3))      # each example, over all its channels and space
instance_norm = normalize(activations, axes=(2, 3))      # each example and channel, over space

print("batch norm:    channel 0 across the batch has mean", round(batch_norm[:, 0].mean(), 3) + 0.0, "std", round(batch_norm[:, 0].std(), 3))
print("layer norm:    example 0 has mean", round(layer_norm[0].mean(), 3), "std", round(layer_norm[0].std(), 3))
print("instance norm: example 0, channel 0 has mean", round(instance_norm[0, 0].mean(), 3) + 0.0, "std", round(instance_norm[0, 0].std(), 3))

* **Batch norm:** CNNs with reasonable batch sizes.
* **Layer norm:** behaves the same in training and testing; **used in RNNs and Transformers**.
* **Instance norm:** style transfer. **Group norm:** groups of channels; good with small batches (detection, segmentation).

## 8. In PyTorch (optional, run in Colab)

In [ ]:
# [PyTorch] Needs: pip install torch
import torch, torch.nn as nn
conv = nn.Conv2d(in_channels=3, out_channels=10, kernel_size=5, stride=1, padding=2)
print("parameters:", sum(p.numel() for p in conv.parameters()), "  output:", tuple(conv(torch.randn(1, 3, 32, 32)).shape))

lenet = nn.Sequential(
    nn.Conv2d(1, 20, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(20, 50, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(2450, 500), nn.ReLU(), nn.Linear(500, 10))
print("LeNet-5 output:", tuple(lenet(torch.randn(4, 1, 28, 28)).shape))

## Summary
* A **convolution** slides a small filter over the image: local connections + shared weights = few parameters.
* **Output size** $= (W - K + 2P)/S + 1$; padding keeps the size, stride shrinks it.
* Put a **ReLU** between convolutions, or they collapse into one.
* **Pooling** downsamples with no parameters and tolerates small shifts.
* The classic CNN: **[Conv, ReLU, Pool] × N → fully connected**. Spatial size goes down, channels go up.
* **Batch norm** normalizes with batch statistics during training and running statistics at test time. Forgetting to switch modes is a classic bug.

### Check your understanding
1. Input 3×32×32, ten 5×5 filters, stride 1, pad 2: output shape and number of parameters?
2. Why does a CNN have far fewer parameters than a fully connected network?
3. What's the receptive field of three stacked 3×3 convolutions?
4. Why does batch norm behave differently at training and test time?

<details><summary>Answers</summary>

1. 10×32×32, and 10 × (3·5·5 + 1) = 760 parameters.
2. Each filter is small, and the same filter is reused at every position.
3. 7×7 (1 + 3·2).
4. Training uses the current batch's statistics; testing uses fixed running averages, so a prediction doesn't depend on the other images in the batch.
</details>

**Next:** Lecture 8, CNN architectures (AlexNet, VGG, ResNet, …).